# 16 从零手写 RAG 原理

目标：不用 LangChain、LlamaIndex、向量库或大模型 API，先把 RAG 的原始链路跑清楚：原始文档 -> chunk -> 向量化 -> 相似度检索 -> context budget -> prompt -> 基于资料回答 -> 引用检查。

这个 notebook 追求可解释，不追求生产效果。真实项目可以把每个手写模块替换成 embedding model、vector database、reranker 和 LLM。

## 1. 原始知识库

RAG 的第一步不是模型，而是资料治理。每条资料至少要保留 `id`、`source` 和正文，后面引用、权限、版本、排错都依赖这些 metadata。

In [ ]:
documents = [
    {
        "id": "kv-cache",
        "source": "deployment-notes.md#kv-cache",
        "text": "KV cache 保存每层 attention 的 key 和 value。prefill 阶段会为整段 prompt 建立缓存，decode 阶段每生成一个 token 追加一格缓存。长上下文和高并发会显著增加显存占用。",
    },
    {
        "id": "ttft-tpot",
        "source": "deployment-notes.md#latency",
        "text": "TTFT 是 time to first token，表示首 token 延迟，主要受排队、tokenize、prefill 和调度影响。TPOT 是 time per output token，常用于衡量 decode 阶段每个输出 token 的速度。",
    },
    {
        "id": "quantization",
        "source": "deployment-notes.md#quantization",
        "text": "量化把权重从 FP16 或 FP32 降到 INT8、INT4 等格式，通常降低显存占用。速度是否提升取决于硬件 kernel、batch 大小和反量化开销。",
    },
    {
        "id": "lora",
        "source": "deployment-notes.md#lora",
        "text": "LoRA 冻结基础模型，只训练低秩 adapter。它能降低微调显存和保存成本，但线上需要管理 adapter 加载、合并和多租户隔离。",
    },
]

for doc in documents:
    print(doc["id"], "=>", doc["source"])
    print(doc["text"])
    print()

## 2. 切 chunk

RAG 通常不会按整篇文档检索，而是把文档切成 chunk。chunk 太大，召回粗糙且浪费上下文；chunk 太小，语义容易断。这里用字符长度做最小演示，真实项目更常按 token、标题层级、段落或语义边界切。

In [ ]:
import re


def split_sentences(text):
    pieces = re.split(r"(?<=[。！？.!?])", text)
    return [piece.strip() for piece in pieces if piece.strip()]


def chunk_document(doc, max_chars=72, overlap_sentences=1):
    sentences = split_sentences(doc["text"])
    chunks = []
    current = []
    current_len = 0

    for sentence in sentences:
        if current and current_len + len(sentence) > max_chars:
            chunks.append("".join(current))
            current = current[-overlap_sentences:] if overlap_sentences else []
            current_len = sum(len(item) for item in current)
        current.append(sentence)
        current_len += len(sentence)

    if current:
        chunks.append("".join(current))

    rows = []
    for index, text in enumerate(chunks):
        rows.append({"chunk_id": f"{doc['id']}#{index}", "doc_id": doc["id"], "source": doc["source"], "text": text})
    return rows


chunks = []
for doc in documents:
    chunks.extend(chunk_document(doc))

for chunk in chunks:
    print(chunk["chunk_id"], chunk["source"])
    print(chunk["text"])
    print("chars=", len(chunk["text"]))
    print()

## 3. 手写最朴素的向量化

真实 RAG 会用 embedding 模型把文本变成稠密向量。为了理解原理，这里先做词袋向量：每个维度代表一个 token，值是出现次数。它很原始，但足够说明“向量检索 = 把 query 和 chunk 放到同一个向量空间里比相似度”。

In [ ]:
import math
from collections import Counter


def tokenize(text):
    # 中文按单字切，英文/数字按词切。这里只为教学透明，不代表最佳中文分词。
    return re.findall(r"[\u4e00-\u9fff]|[A-Za-z0-9_]+", text.lower())


def build_vocabulary(texts):
    vocab = {}
    for text in texts:
        for token in tokenize(text):
            if token not in vocab:
                vocab[token] = len(vocab)
    return vocab


def vectorize(text, vocab):
    counts = Counter(tokenize(text))
    vector = [0.0] * len(vocab)
    for token, count in counts.items():
        if token in vocab:
            vector[vocab[token]] = float(count)
    return vector


def cosine_similarity(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(y * y for y in b))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)


vocab = build_vocabulary([chunk["text"] for chunk in chunks])
chunk_vectors = {chunk["chunk_id"]: vectorize(chunk["text"], vocab) for chunk in chunks}

print("chunk count =", len(chunks))
print("vocab size =", len(vocab))
print("first 30 vocab tokens =", list(vocab)[:30])

## 4. Query 向量化和 Top-k 检索

用户问题也用同一个 `vocab` 向量化，然后和每个 chunk 算 cosine similarity，分数最高的就是检索结果。生产系统会把这里换成 ANN 索引、metadata filter、hybrid search 和 reranker。

In [ ]:
def retrieve(query, k=3):
    query_vector = vectorize(query, vocab)
    scored = []
    for chunk in chunks:
        score = cosine_similarity(query_vector, chunk_vectors[chunk["chunk_id"]])
        scored.append((score, chunk))
    scored.sort(key=lambda item: item[0], reverse=True)
    return scored[:k]


question = "TTFT 和 TPOT 分别是什么意思，受什么影响？"
hits = retrieve(question, k=3)

for rank, (score, chunk) in enumerate(hits, start=1):
    print(f"rank={rank} score={score:.3f} chunk={chunk['chunk_id']} source={chunk['source']}")
    print(chunk["text"])
    print()

## 5. 组装 context 和 prompt

检索到的内容不能无脑全塞进 prompt。RAG 需要一个 context budget：按分数、来源、多样性和长度取舍。这里用字符数模拟 token budget。

In [ ]:
def assemble_context(hits, max_context_chars=420):
    blocks = []
    used = 0
    for rank, (score, chunk) in enumerate(hits, start=1):
        block = f"[{rank}] source={chunk['source']} score={score:.3f}\n{chunk['text']}"
        if used + len(block) > max_context_chars:
            continue
        blocks.append(block)
        used += len(block)
    return "\n\n".join(blocks)


def build_prompt(question, hits):
    context = assemble_context(hits)
    return (
        "你是一个严谨的问答助手。只能根据资料回答；资料不足时说不知道；回答必须带引用编号。\n\n"
        f"资料：\n{context}\n\n"
        f"问题：{question}\n"
        "答案："
    )


prompt = build_prompt(question, hits)
print(prompt)

## 6. 用规则模拟生成

真实系统这里会调用 LLM。为了让 notebook 离线可跑，我们用一个很粗糙的规则函数模拟“只根据检索资料回答”：从命中的 chunk 里挑和问题重叠最多的句子，并补引用编号。重点是看清楚生成阶段拿到的不是整个知识库，而是检索阶段交给它的有限 context。

In [ ]:
def sentence_overlap_score(question, sentence):
    q_tokens = set(tokenize(question))
    s_tokens = set(tokenize(sentence))
    return len(q_tokens & s_tokens)


def generate_grounded_answer(question, hits, max_sentences=3):
    candidates = []
    for rank, (_, chunk) in enumerate(hits, start=1):
        for sentence in split_sentences(chunk["text"]):
            overlap = sentence_overlap_score(question, sentence)
            if overlap:
                candidates.append((overlap, rank, sentence))
    candidates.sort(key=lambda item: item[0], reverse=True)

    if not candidates:
        return "根据已检索到的资料，我不知道。"

    used = []
    seen_sentences = set()
    for _, rank, sentence in candidates:
        if sentence in seen_sentences:
            continue
        used.append(f"{sentence}[{rank}]")
        seen_sentences.add(sentence)
        if len(used) >= max_sentences:
            break
    return "".join(used)


answer = generate_grounded_answer(question, hits)
print(answer)

## 7. 引用检查

RAG 不是“有引用就一定对”，但引用检查是底线：答案里出现的引用编号必须存在。更严格的系统还会做 groundedness、answer correctness、context precision、context recall。

In [ ]:
def extract_citation_numbers(answer):
    return [int(item) for item in re.findall(r"\[(\d+)\]", answer)]


def check_citations(answer, hits):
    valid = set(range(1, len(hits) + 1))
    cited = extract_citation_numbers(answer)
    missing = [number for number in cited if number not in valid]
    return {"cited": cited, "missing": missing, "ok": bool(cited) and not missing}


print(check_citations(answer, hits))
bad_answer = "TTFT 主要由数据库事务锁决定。[9]"
print(check_citations(bad_answer, hits))

## 8. 失败案例和 query rewrite

最原始的词袋检索不懂同义词和语义。比如用户问“首字延迟”，如果资料里写的是 `TTFT` 和 `first token`，检索可能不稳定。生产里常用 dense embedding、query rewrite、hybrid search、reranker 来缓解。

In [ ]:
hard_question = "首字延迟和逐字输出速度怎么优化？"

synonyms = {
    "首字延迟": "TTFT first token 首 token 延迟 prefill 排队 tokenize 调度",
    "逐字输出速度": "TPOT time per output token decode 输出 token 速度",
    "低成本微调": "LoRA adapter 冻结 低秩 微调 显存 保存成本",
}


def rewrite_query(query):
    expanded = [query]
    for phrase, expansion in synonyms.items():
        if phrase in query:
            expanded.append(expansion)
    return " ".join(expanded)


for label, query in [("original", hard_question), ("rewritten", rewrite_query(hard_question))]:
    print("===", label, "===")
    print(query)
    hits_for_query = retrieve(query, k=3)
    for rank, (score, chunk) in enumerate(hits_for_query, start=1):
        print(f"rank={rank} score={score:.3f} chunk={chunk['chunk_id']}")
    print(generate_grounded_answer(hard_question, hits_for_query))
    print()

## 9. 手写模块和框架怎么对应

学原理时先手写，做工程时再按模块替换。框架不是 RAG 的本质，它们主要帮你把这些步骤标准化、可组合、可观测。

| 手写步骤 | 工程里常见替换 | 解决的问题 |
|---|---|---|
| document metadata | 文档解析器、权限系统、对象存储 | 保留来源、权限、版本和可追溯性 |
| chunk_document | LangChain splitter、LlamaIndex node parser、Haystack preprocessor | 稳定切分、overlap、按标题或 token 控制粒度 |
| vectorize | embedding model，例如 bge、gte、text-embedding 系列 | 从词面匹配升级到语义匹配 |
| chunk_vectors | FAISS、Chroma、Qdrant、Milvus、pgvector、Weaviate | 向量索引、过滤、持久化、扩展和高并发检索 |
| retrieve | retriever、hybrid search、metadata filter | Top-k 召回、关键词和语义混合检索 |
| rerank | cross-encoder reranker、bge-reranker | 把召回结果重新精排，降低错 chunk 进入 prompt 的概率 |
| assemble_context | prompt template、context compressor | 控制上下文预算，去重，保留引用 |
| generate_grounded_answer | LLM、tool calling、structured output | 根据证据生成答案，约束格式和引用 |
| check_citations | RAGAS、DeepEval、自定义 groundedness check | 评估引用、忠实度、召回和答案正确性 |

常见框架选择可以这样理解：LangChain/LangGraph 偏应用编排和 agent workflow；LlamaIndex 偏数据索引、检索和文档问答；Haystack 偏 pipeline 风格；RAGFlow 偏开箱即用的文档 RAG 平台；DSPy 偏把检索和生成流程当成可优化程序。


## 面试总结

- RAG = indexing pipeline + retrieval pipeline + generation pipeline + evaluation pipeline，不只是“接一个向量库”。
- 最小链路：load documents -> chunk -> embed -> index -> retrieve -> assemble context -> generate -> cite -> evaluate。
- chunk 决定召回颗粒度，embedding 决定语义匹配能力，reranker 决定 top-k 精排质量，context budget 决定最终喂给 LLM 的证据。
- 生成错误不一定是模型差，也可能是没召回、召回了错误 chunk、上下文超预算、引用错配或资料本身过期。
- 学原理时先手写；做工程时再替换为 embedding model、FAISS/Chroma/Qdrant/Milvus/pgvector、reranker、LLM 和 RAGAS 等评测工具。